# ComicsPAP sequence filling: experiment runner

This notebook only **launches the scripts in `scripts/`**; all logic lives there. Run the cells top to bottom, or pick the section you need. Every long job is resumable (finished runs / cached features are skipped).

**Three Python environments are used** (see the README for the install commands):

| name | used for | key packages |
|---|---|---|
| `main` (the kernel running this notebook) | data prep, visual features, text embeddings, fusion models, plots | torch, transformers, sentence-transformers, matplotlib |
| `ocr` | PaddleOCR (PP-OCRv6) and the OCR spot-check | paddlepaddle-gpu, paddleocr, pillow |
| `qwen` | Qwen2.5-VL LoRA training / evaluation | torch, transformers, peft, bitsandbytes, qwen-vl-utils |

Edit **only the configuration cell** below.

## 0. Configuration

In [ ]:
import os, sys
from pathlib import Path

# ---- EDIT THESE ------------------------------------------------------------
RAW_DATA   = "/path/to/ComicsPAP/sequence_filling"   # folder with train/ validation/ test/ parquet files
DATA_ROOT  = "/path/to/work_dir"                      # extracted panels, manifests, OCR, embeddings go here
PY_OCR     = "/path/to/paddle_env/bin/python"         # Windows: r"C:\envs\paddle_env\Scripts\python.exe"
PY_QWEN    = "/path/to/qwen_env/bin/python"           # Windows: r"C:\envs\qwen_env\Scripts\python.exe"
FONT       = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"   # Windows: C:/Windows/Fonts/arial.ttf (the released baseline used Arial)
# Optional (e.g. behind a firewall): uncomment to use a Hugging Face mirror / custom cache
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["HF_HOME"]     = "/path/to/hf_cache"
# -----------------------------------------------------------------------------

PY_MAIN = sys.executable
MAN     = f"{DATA_ROOT}/manifests"
OCR_DIR = f"{DATA_ROOT}/ocr/ppocrv6_medium"
EMB     = f"{DATA_ROOT}/embeddings"
TEXT_EMB = f"{EMB}/text_ppocrv6_medium"
VIS = {"dinov2": (f"{EMB}/visual", 768),
       "clip":   (f"{EMB}/visual_clip", 512),
       "siglip": (f"{EMB}/visual_siglip", 768)}

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
assert Path("scripts").is_dir(), "start Jupyter from the repository root (the folder that contains scripts/)"
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)

## 1. Dataset: inspect, extract panels, build manifests
`01` prints the parquet schema; **set the column names in `configs/dataset.yaml` from its output** (see the README). `02` streams the parquet files and writes one JPEG per panel plus `manifests/<split>.jsonl`.

In [ ]:
!{PY_MAIN} scripts/01_inspect_dataset.py --data_root "{RAW_DATA}"

In [ ]:
!{PY_MAIN} scripts/02_prepare_manifest.py --data_root "{RAW_DATA}" --out_root "{DATA_ROOT}" --config configs/dataset.yaml --on_mismatch keep

A few samples have a non-standard layout (not 4 context + 5 candidates). Keep the standard ones in the manifests and set the others aside (`<split>_nonstandard.jsonl`). The first backup `*.jsonl.bak` is never overwritten, so the cell can be re-run.

In [ ]:
import json, shutil
from collections import Counter

N_CTX, N_CAND = 4, 5
for s in ["train", "validation", "test"]:
    p = Path(MAN) / f"{s}.jsonl"
    bak = p.with_suffix(".jsonl.bak")
    if not bak.exists():
        shutil.copy(p, bak)
    odd = p.with_name(f"{s}_nonstandard.jsonl")
    kept = dropped = 0
    with open(bak, encoding="utf-8") as fin, open(p, "w", encoding="utf-8") as fout, open(odd, "w", encoding="utf-8") as fodd:
        for line in fin:
            r = json.loads(line)
            if len(r["context_panels"]) == N_CTX and len(r["candidates"]) == N_CAND:
                fout.write(line); kept += 1
            else:
                fodd.write(line); dropped += 1
    print(f"{s}: kept {kept}, set aside {dropped}")

# sanity checks: layout and label distribution (labels are hidden = -1 in test)
for s in ["train", "validation", "test"]:
    rows = [json.loads(l) for l in open(Path(MAN) / f"{s}.jsonl", encoding="utf-8")]
    print(s, "layouts", dict(Counter((len(r["context_panels"]), len(r["candidates"])) for r in rows)),
          "| labels", sorted(Counter(r["label"] for r in rows).items()))

## 2. Frozen visual features (main env)
One output folder per encoder. Run for validation, test **and** train with the same `--out_dir`. The printed embedding dimension is what you pass as `--visual_dim` later (DINOv2 768, CLIP ViT-B/16 512, SigLIP ViT-B/16 768).

In [ ]:
for s in ["validation", "test", "train"]:
    for enc, (out_dir, _) in VIS.items():
        print(f"=== {s} / {enc} ===")
        !{PY_MAIN} scripts/06_extract_visual_features.py --encoder {enc} --manifest "{MAN}/{s}.jsonl" --data_root "{DATA_ROOT}" --out_dir "{out_dir}" --batch_size 8 --fp16

## 3. Visual-only fusion baselines (frozen features + small transformer)
A contiguous block of the **end of train** is held out (with a gap) for checkpoint selection; the official validation split is never used for selection.

In [ ]:
# quick smoke test (1 epoch)
!{PY_MAIN} scripts/08_train_fusion.py --no_text --seed 0 --epochs 1 --out outputs/smoke --manifest_dir "{MAN}" --visual_dir "{VIS['dinov2'][0]}" --data_root "{DATA_ROOT}"

In [ ]:
# 3 encoders x 3 seeds (DINOv2 also with the stronger-regularisation preset "reg")
feats = " ".join(f"{k}={d}@{dim}" for k, (d, dim) in VIS.items())
!{PY_MAIN} scripts/11_run_sweep.py --manifest_dir "{MAN}" --features {feats} --configs base --seeds 0 1 2 --out_root outputs/sweep_visual --data_root "{DATA_ROOT}"
!{PY_MAIN} scripts/11_run_sweep.py --manifest_dir "{MAN}" --features dinov2={VIS['dinov2'][0]}@768 --configs reg --seeds 0 1 2 --out_root outputs/sweep_visual --data_root "{DATA_ROOT}"

In [ ]:
# summary table only (no training)
!{PY_MAIN} scripts/11_run_sweep.py --aggregate_only --out_root outputs/sweep_visual

## 4. OCR (PP-OCRv6) and text features
OCR runs in the **`ocr` env** (no torch needed). It is cached per unique image, so the command can be repeated after an interruption.

In [ ]:
# sanity check on one image: prints the raw result keys and the parsed detections
import glob
img = sorted(glob.glob(f"{DATA_ROOT}/extracted_panels/validation/*/context_0.jpg"))[0]
!{PY_OCR} scripts/04_run_ocr.py --inspect --image "{img}"

In [ ]:
!{PY_OCR} scripts/04_run_ocr.py --data_root "{DATA_ROOT}" --out_dir "{OCR_DIR}" --manifest "{MAN}/validation.jsonl" "{MAN}/test.jsonl" "{MAN}/train.jsonl"

**Optional: OCR spot-check.** Builds an HTML page to label 60 random panels (blind transcription, then detection judgments), then scores CER/WER and detection precision/recall for several confidence thresholds. Use `--n 150` for stable numbers.

In [ ]:
!{PY_OCR} scripts/04b_ocr_spotcheck.py sample --ocr_dir "{OCR_DIR}" --split validation --data_root "{DATA_ROOT}" --out_dir "{DATA_ROOT}/ocr/spotcheck" --n 60
# ... open {DATA_ROOT}/ocr/spotcheck/index.html, label, download labels.json into that folder, then:
# !{PY_OCR} scripts/04b_ocr_spotcheck.py score --labels "{DATA_ROOT}/ocr/spotcheck/labels.json"

In [ ]:
# MiniLM sentence embeddings of the OCR text of every panel (main env)
!{PY_MAIN} scripts/05_generate_text_embeddings.py --ocr_dir "{OCR_DIR}" --manifest "{MAN}/validation.jsonl" "{MAN}/test.jsonl" "{MAN}/train.jsonl" --out_dir "{TEXT_EMB}"

In [ ]:
# visual-only vs visual+OCR vs OCR-only (SigLIP features, 3 seeds)
sig = f"siglip={VIS['siglip'][0]}@768"
!{PY_MAIN} scripts/11_run_sweep.py --manifest_dir "{MAN}" --features {sig} --text_dir "{TEXT_EMB}" --modes visual visual_text text --configs base --seeds 0 1 2 --out_root outputs/sweep_text --data_root "{DATA_ROOT}"
# ways of using the text: text dropout / similarity features only / both
!{PY_MAIN} scripts/11_run_sweep.py --manifest_dir "{MAN}" --features {sig} --text_dir "{TEXT_EMB}" --modes visual_text --configs tdrop sim both --seeds 0 1 2 --out_root outputs/sweep_text_usage --data_root "{DATA_ROOT}"
# OCR variants: confidence filter x reading order
!{PY_MAIN} scripts/14_run_ocr_ablations.py --ocr_dir "{OCR_DIR}" --manifest_dir "{MAN}" --text_root "{TEXT_EMB}" --features {sig} --out_root outputs/sweep_ocr_ablation --configs base --seeds 0 1 2 --data_root "{DATA_ROOT}"

## 5. Figures and diagnostics
`13` dumps logits for every finished fusion run (needed for calibration plots). `12_visualize.py` only needs matplotlib / numpy / pillow, so it also works in the `ocr` env.

In [ ]:
!{PY_MAIN} scripts/13_dump_logits.py --runs_root outputs/sweep_visual
!{PY_MAIN} scripts/12_visualize.py all --runs_root outputs/sweep_visual --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --ocr_dir "{OCR_DIR}"

In [ ]:
# leakage / distribution shift between train, holdout, validation and test (exact-image and text overlap)
!{PY_MAIN} scripts/12_visualize.py shift --ocr_dir "{OCR_DIR}" --out_dir outputs/figures

## 6. Qwen2.5-VL-3B-Instruct (`qwen` env)
All Qwen commands below use 4-bit weights and `--max_pixels 401408` so they fit an **8 GB GPU** (RTX 3070). With more memory, drop `--load_4bit` / raise `--max_pixels`.

* **E0** zero-shot base model, **E1** released ComicsPAP LoRA adapter (`VLR-CVC/Qwen2.5-VL-3B-Instruct-lora-ComicsPAP`).
* `--decode score` = forced choice over the digits 0..4 (never unparsable); `generate` = free generation + parsing of `answer: N`.

In [ ]:
# smoke test: 3 samples; saves the rendered two-row composite so you can look at it
!{PY_QWEN} scripts/07b_eval_qwen.py --smoke --adapter none --load_4bit --max_pixels 401408 --out outputs/qwen_smoke --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

In [ ]:
# E1: released adapter on the validation split
!{PY_QWEN} scripts/07b_eval_qwen.py --adapter VLR-CVC/Qwen2.5-VL-3B-Instruct-lora-ComicsPAP --load_4bit --max_pixels 401408 --decode score --out outputs/qwen_e1_lowres --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"
# E0: zero-shot base model
!{PY_QWEN} scripts/07b_eval_qwen.py --adapter none --load_4bit --max_pixels 401408 --decode both --out outputs/qwen_e0_lowres --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

In [ ]:
# E1 + OCR text added to the prompt at inference only (adapter was NOT trained with OCR -> expected to hurt)
!{PY_QWEN} scripts/07b_eval_qwen.py --adapter VLR-CVC/Qwen2.5-VL-3B-Instruct-lora-ComicsPAP --load_4bit --max_pixels 401408 --decode score --ocr_dir "{OCR_DIR}" --out outputs/qwen_e1_ocr_infer --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

### Continue fine-tuning the released adapter (images only)
Checkpoints are selected on a train holdout; validation is only scored at the end with the best adapter.

> The exact flags of the *first* stage (`outputs/qwen_cont_images`) were not recorded in the original notebook; the command below uses the same recipe as the OCR run. Adjust it to what you actually ran. Stage 2 resumes from `last_adapter`.

In [ ]:
# stage 1 (images only) -- reconstructed recipe, adjust if needed
!{PY_QWEN} scripts/07_train_qwen.py --init_adapter VLR-CVC/Qwen2.5-VL-3B-Instruct-lora-ComicsPAP --load_4bit --max_pixels 401408 --lr 2e-5 --train_samples 1000 --epochs 2 --grad_accum 8 --eval_every 50 --eval_samples 150 --seed 0 --out outputs/qwen_cont_images --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

In [ ]:
# stage 2: resume from the last adapter of stage 1
!{PY_QWEN} scripts/07_train_qwen.py --init_adapter outputs/qwen_cont_images/last_adapter --load_4bit --max_pixels 401408 --lr 2e-5 --train_samples 1000 --epochs 4 --grad_accum 8 --eval_every 150 --patience 2 --eval_samples 150 --seed 0 --out outputs/qwen_cont_images_continue --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

In [ ]:
# score a saved adapter on the official validation split
!{PY_QWEN} scripts/07_train_qwen_eval.py --adapter outputs/qwen_cont_images/best_adapter --out outputs/qwen_cont_images/val_eval --load_4bit --max_pixels 401408 --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}" --final_decode score

### E8: fine-tune with OCR text in the prompt

In [ ]:
!{PY_QWEN} scripts/07_train_qwen.py --init_adapter VLR-CVC/Qwen2.5-VL-3B-Instruct-lora-ComicsPAP --load_4bit --max_pixels 401408 --lr 2e-5 --train_samples 1000 --epochs 2 --grad_accum 8 --eval_every 50 --eval_samples 150 --seed 0 --ocr_dir "{OCR_DIR}" --out outputs/qwen_cont_ocr --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --font "{FONT}"

In [ ]:
# training curves + health check, and "what does the model actually predict" pictures
!{PY_MAIN} scripts/12_visualize.py history --run_dir outputs/qwen_cont_ocr
!{PY_MAIN} scripts/12_visualize.py predictions --run_dir outputs/qwen_cont_images/val_eval --split val --manifest_dir "{MAN}" --data_root "{DATA_ROOT}" --n 8 --which mixed